In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "config.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find config.py. Start Jupyter from the repository or its Notebooks folder."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from config import (
    DAILY_FLOW_FILE,
    DELIVERY_THRESHOLD,
    FORECAST_14D_FILE,
    FORECAST_PERFORMANCE_FILE,
    FORECAST_RISK_LEGACY_FILE,
    HOLDOUT_WEEKS,
    HORIZON_WEEKS,
    ensure_output_directories,
)

df = pd.read_csv(DAILY_FLOW_FILE)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["product_id", "Date"])

test_days = HOLDOUT_WEEKS * 7
forecast_days = HORIZON_WEEKS * 7
cutoff = df["Date"].max() - pd.Timedelta(days=test_days)

train = df[df["Date"] <= cutoff].copy()
test = df[df["Date"] > cutoff].copy()

print("Train end:", train["Date"].max())
print("Test start:", test["Date"].min())
print("Test end:", test["Date"].max())


In [ ]:
def forecast_demand(history, horizon, model):
    history = np.asarray(history, dtype=float)

    if model == "seasonal_naive_7":
        last_7 = history[-7:]
        return np.resize(last_7, horizon)

    if model == "moving_average_7":
        return np.repeat(history[-7:].mean(), horizon)

    if model == "moving_average_28":
        return np.repeat(history[-28:].mean(), horizon)

    raise ValueError("Unknown model")

In [ ]:
models = [
    "seasonal_naive_7",
    "moving_average_7",
    "moving_average_28"
]

performance = []

for product_id in df["product_id"].unique():

    history = (
        train.loc[
            train["product_id"] == product_id,
            "sales_units"
        ]
        .sort_index()
        .to_numpy()
    )

    actual = (
        test.loc[
            test["product_id"] == product_id,
            "sales_units"
        ]
        .to_numpy()
    )

    for model in models:
        prediction = forecast_demand(
            history,
            len(actual),
            model
        )

        mae = np.mean(np.abs(actual - prediction))

        if actual.sum() > 0:
            wape = (
                np.abs(actual - prediction).sum()
                / actual.sum()
            )

            bias = (
                prediction.sum() - actual.sum()
            ) / actual.sum()
        else:
            wape = np.nan
            bias = np.nan

        performance.append({
            "product_id": product_id,
            "model": model,
            "mae": mae,
            "wape": wape,
            "bias": bias
        })

performance = pd.DataFrame(performance)

In [ ]:
performance["selection_score"] = np.where(
    performance["wape"].notna(),
    performance["wape"],
    performance["mae"]
)

best_models = (
    performance
    .sort_values(["product_id", "selection_score"])
    .groupby("product_id", as_index=False)
    .first()
)

best_models[
    ["product_id", "model", "mae", "wape", "bias"]
].head(10)

In [ ]:
print(
    performance.groupby("model")[
        ["mae", "wape", "bias"]
    ].mean()
)

In [ ]:
future_dates = pd.date_range(
    df["Date"].max() + pd.Timedelta(days=1),
    periods=forecast_days,
    freq="D",
)

future_rows = []
for product_id in df["product_id"].unique():
    history = df.loc[
        df["product_id"] == product_id,
        "sales_units",
    ].to_numpy()
    selected_model = best_models.loc[
        best_models["product_id"] == product_id,
        "model",
    ].iloc[0]
    prediction = forecast_demand(history, forecast_days, selected_model)

    for date, value in zip(future_dates, prediction):
        future_rows.append({
            "Date": date,
            "product_id": product_id,
            "forecast_sales_units": max(0, value),
            "selected_model": selected_model,
        })

forecast_14d = pd.DataFrame(future_rows)


In [ ]:
product_dimension = (
    df[
        ["product_id", "Group", "Sub-Group"]
    ]
    .drop_duplicates()
)

forecast_14d = forecast_14d.merge(
    product_dimension,
    on="product_id",
    how="left"
)

In [ ]:
recent_start = df["Date"].max() - pd.Timedelta(days=test_days - 1)
recent = df[df["Date"] >= recent_start]

recent_summary = (
    recent.groupby(["product_id", "Group", "Sub-Group"], as_index=False)[
        ["sales_units", "production_units", "delivery_units"]
    ].sum()
)
future_summary = (
    forecast_14d.groupby(
        ["product_id", "Group", "Sub-Group"],
        as_index=False,
    )["forecast_sales_units"].sum()
)


In [ ]:
risk = future_summary.merge(
    recent_summary,
    on=["product_id", "Group", "Sub-Group"],
    how="left",
)
risk["projected_14d_production"] = (
    risk["production_units"] / test_days * forecast_days
)
risk["forecast_production_gap"] = (
    risk["forecast_sales_units"] - risk["projected_14d_production"]
)
risk["recent_delivery_ratio"] = np.where(
    risk["sales_units"] > 0,
    risk["delivery_units"] / risk["sales_units"],
    np.nan,
)


In [ ]:
conditions = [
    (
        (risk["forecast_production_gap"] > 0)
        & (risk["recent_delivery_ratio"] < DELIVERY_THRESHOLD)
    ),
    (
        (risk["forecast_production_gap"] > 0)
        | (risk["recent_delivery_ratio"] < DELIVERY_THRESHOLD)
    ),
]
risk["risk_level"] = np.select(
    conditions,
    ["High", "Medium"],
    default="Low",
)


In [ ]:
ensure_output_directories()
performance.to_csv(FORECAST_PERFORMANCE_FILE, index=False)
forecast_14d.to_csv(FORECAST_14D_FILE, index=False)
risk.to_csv(FORECAST_RISK_LEGACY_FILE, index=False)

print(f"Saved: {FORECAST_PERFORMANCE_FILE}")
print(f"Saved: {FORECAST_14D_FILE}")
print(f"Saved: {FORECAST_RISK_LEGACY_FILE}")


In [ ]:
print(forecast_14d.shape)
print(forecast_14d["product_id"].nunique())
print(forecast_14d["Date"].min())
print(forecast_14d["Date"].max())

print(
    risk["risk_level"].value_counts()
)

risk.sort_values(
    "forecast_production_gap",
    ascending=False
).head(10)